# Circuit Visualization: Hardware-Efficient Ansatz

Visualizes the `RY`/`RZ` + linear-CNOT ansatz used in the QNN models,
layerwise construction, the local vs global cost operators, and parameter
counts by architecture.

In [ ]:
import os
import sys
from pathlib import Path

# Run from the project root regardless of the notebook's directory.
ROOT = Path.cwd()
while not (ROOT / "configs").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

os.environ.setdefault("TF_USE_LEGACY_KERAS", "1")


## 1. Basic Circuit Structure

In [ ]:
import cirq

from src.models import QuantumCircuit

qc = QuantumCircuit(n_qubits=8, n_layers=2)
print(f"Parameters: {len(qc.get_parameters())}")
print()
print(qc.visualize())


## 2. Ansatz Layer Structure

Each layer: `RY` + `RZ` per qubit, then a
linear CNOT chain between neighbouring qubits.

In [ ]:
print(qc.circuit)


## 3. Layerwise Construction

Used by the layerwise approach: the circuit up
to layer `k` is built incrementally.

In [ ]:
qc_lw = QuantumCircuit(n_qubits=8, n_layers=4)
for k in range(4):
    sub = qc_lw.get_circuit_up_to_layer(k)
    print(f"Up to layer {k}: {len(sub)} moments, "
          f"{len(qc_lw.get_layer_parameters(k))} params in layer {k}")


## 4. Circuit Depth Analysis

In [ ]:
depths = [2, 4, 6, 8]
for depth in depths:
    c = QuantumCircuit(n_qubits=8, n_layers=depth).get_circuit()
    print(f"depth {depth}: {len(c)} moments")


## 5. Entanglement Structure

In [ ]:
cnots = sum(1 for op in qc.get_circuit().all_operations()
            if isinstance(op.gate, cirq.CNotPowGate))
print(f"CNOT gates in depth-2, 8-qubit ansatz: {cnots}")


## 6. Local vs Global Cost Operators

In [ ]:
from src.models import create_readout_operators

global_op = create_readout_operators(8, local=False)
local_op = create_readout_operators(8, local=True)
print("Global cost (Z-tensor product):")
print(" ", global_op)
print("Local cost ((1/n) sum Z_i):")
print(" ", local_op)


## 7. Parameter Count by Architecture

`P = 2 * n_qubits * n_layers`.

In [ ]:
import numpy as np

qubit_counts = [2, 4, 6, 8]
layer_counts = [2, 4, 6, 8]
matrix = np.zeros((len(qubit_counts), len(layer_counts)), dtype=int)
for i, nq in enumerate(qubit_counts):
    for j, nl in enumerate(layer_counts):
        matrix[i, j] = 2 * nq * nl
print("rows = qubits, cols = layers")
print(matrix)


## 8. Circuit Simulation Example

In [ ]:
import sympy

sim = cirq.Simulator()
symbols = qc.get_parameters()
values = np.random.RandomState(0).uniform(-0.05, 0.05, size=len(symbols))
resolver = cirq.ParamResolver({s: v for s, v in zip(symbols, values)})
result = sim.simulate(qc.get_circuit(), param_resolver=resolver)
print(f"Final state dimension: {result.final_state_vector.shape}")


## 9. Conclusion

- Parameters scale linearly: `P = 2 n L`.
- Global cost is a product observable; local cost is a scaled sum of
  single-qubit `Z` terms; both are single `cirq.PauliSum` operators.